# α-Sensitivity Analysis — BM25 + TAS-B Interpolation

This notebook visualises how the interpolation parameter α affects the hybrid ranking pipeline  
`BM25 → TAS-B → FFInterpolate(α)` across four BEIR datasets.

**Plots produced:**
1. Per-metric area charts (RR@10, nDCG@10, AP@100) — all four datasets on one canvas
2. Normalised performance heatmap — highlights the "safe zone" of α values
3. Robustness band plot — mean ± std across datasets at each α
4. Regret plot — how much each dataset loses vs. its own optimum when a global α is applied


In [24]:
fine_tuned_alpha_value = 0.35 # value taken form rq2, step 2

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Use a clean white template as base; we'll style on top
pio.templates.default = "plotly_white"

# ── Colour palette (colour-blind-friendly, print-safe) ──
COLORS = {
    "FiQA":     "#2D6A4F",   # deep green
    "SciFact":  "#E07A5F",   # terracotta
    "ArguAna":  "#3D5A80",   # slate blue
    "NFCorpus": "#9B5DE5",   # violet
}

METRICS = ["RR@10", "nDCG@10", "AP@100"]


In [26]:
df = pd.read_csv(Path.cwd() / "alpha_sweep_results.csv")

# Quick sanity check
print(f"Shape: {df.shape}")
print(f"Datasets: {df['dataset'].unique()}")
print(f"Alpha range: {df['alpha'].min():.2f} – {df['alpha'].max():.2f}")
df.head()


Shape: (404, 5)
Datasets: ['FiQA' 'SciFact' 'ArguAna' 'NFCorpus']
Alpha range: 0.00 – 1.00


,dataset,alpha,RR@10,nDCG@10,AP@100
0,FiQA,0.00,0.369461,0.305473,0.252348
1,FiQA,0.01,0.370710,0.306497,0.253109
2,FiQA,0.02,0.371681,0.306984,0.253525
3,FiQA,0.03,0.373727,0.308165,0.255066
4,FiQA,0.04,0.375404,0.308853,0.256113


## 1 · Per-Metric Area Charts

Each chart shows how a single metric responds to α for all four datasets.  
The filled area makes it easy to see overlap and divergence.


In [27]:
def make_area_chart(df, metric, colors):
    fig = go.Figure()
    
    for dataset in df["dataset"].unique():
        sub = df[df["dataset"] == dataset].sort_values("alpha")
        fig.add_trace(go.Scatter(
            x=sub["alpha"], y=sub[metric],
            mode="lines",
            name=dataset,
            line=dict(color=colors[dataset], width=2.5),
            fill="tozeroy",
            fillcolor=colors[dataset].replace(")", ",0.12)").replace("rgb", "rgba")
                      if "rgb" in colors[dataset]
                      else f"rgba({int(colors[dataset][1:3],16)},{int(colors[dataset][3:5],16)},{int(colors[dataset][5:7],16)},0.12)",
            hovertemplate=f"<b>{dataset}</b><br>α=%{{x:.2f}}<br>{metric}=%{{y:.4f}}<extra></extra>",
        ))

    fig.update_layout(
        title=dict(text=f"<b>{metric}</b> vs. Interpolation Weight α", font_size=18),
        xaxis=dict(title="α  (0 = pure TAS-B,  1 = pure BM25)", dtick=0.1, range=[0, 1]),
        yaxis=dict(title=metric),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        template="plotly_white",
        width=820, height=460,
        margin=dict(t=80, b=60),
        font=dict(family="Helvetica Neue, Segoe UI, sans-serif"),
    )
    return fig


for m in METRICS:
    fig = make_area_chart(df, m, COLORS)
    fig.show()


## 1b · Metric Delta vs. Pure BM25

Since datasets have very different absolute scores, the area charts above make it hard to see *where the interpolation helps most*.  
These charts show **Δ = score(α) − score(α=1)**, i.e. the gain (or loss) over pure BM25.  
A positive value means the hybrid pipeline outperforms BM25 alone; the peak shows each dataset's optimal α.


In [28]:
def make_delta_chart(df, metric, colors):
    fig = go.Figure()

    for dataset in df["dataset"].unique():
        sub = df[df["dataset"] == dataset].sort_values("alpha")
        # α = 1.0 is pure BM25 (weight 1 on lexical, 0 on semantic)
        bm25_score = sub.loc[sub["alpha"] == 1.0, metric].values[0]
        delta = sub[metric] - bm25_score

        # Find the peak
        peak_idx = delta.idxmax()
        peak_alpha = sub.loc[peak_idx, "alpha"]
        peak_delta = delta.loc[peak_idx]

        c = colors[dataset]
        rgba_fill = f"rgba({int(c[1:3],16)},{int(c[3:5],16)},{int(c[5:7],16)},0.12)"

        fig.add_trace(go.Scatter(
            x=sub["alpha"], y=delta,
            mode="lines",
            name=dataset,
            line=dict(color=c, width=2.5),
            fill="tozeroy",
            fillcolor=rgba_fill,
            hovertemplate=f"<b>{dataset}</b><br>α=%{{x:.2f}}<br>Δ{metric}=%{{y:+.4f}}<extra></extra>",
        ))

        # Mark the peak with a dot
        fig.add_trace(go.Scatter(
            x=[peak_alpha], y=[peak_delta],
            mode="markers+text",
            marker=dict(color=c, size=8, line=dict(color="white", width=1.5)),
            text=[f"α={peak_alpha:.2f}"],
            textposition="top center",
            textfont=dict(size=10, color=c),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Zero reference line
    fig.add_hline(y=0, line_dash="dot", line_color="#999", line_width=1)

    fig.update_layout(
        title=dict(text=f"<b>Δ {metric}</b> relative to pure BM25 (α=1)", font_size=18),
        xaxis=dict(title="α  (0 = pure TAS-B,  1 = pure BM25)", dtick=0.1, range=[0, 1]),
        yaxis=dict(title=f"Δ {metric}  (positive = better than BM25)"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        template="plotly_white",
        width=820, height=460,
        margin=dict(t=80, b=60),
        font=dict(family="Helvetica Neue, Segoe UI, sans-serif"),
    )
    return fig


for m in METRICS:
    fig = make_delta_chart(df, m, COLORS)
    fig.show()


## 2 · Normalised Performance Heatmap

Each cell shows performance as a **percentage of the dataset's own maximum** for that metric.  
This lets us compare across datasets on a common 0–100 % scale and spot the α "sweet spot" that works well everywhere.


## 3 · Cross-Dataset Robustness Band

For each α, we compute the **mean and ±1 std** of the normalised scores across all four datasets.  
A narrow band means the α value works consistently; a wide band means it helps some datasets but hurts others.

Two vertical lines are drawn:
- **Green dashed** — α with the **highest worst-case** normalised score (most *robust* global choice, derived from test data).
- **Orange dotted** — `fine_tuned_alpha_value` tuned on the **training split**, showing how well the train-tuned choice aligns with the test-set optimum.

In [29]:
def make_heatmap(df, metric, colors):
    datasets = df["dataset"].unique()
    alphas = sorted(df["alpha"].unique())

    # Build matrix: rows = datasets, cols = alphas, values = % of row-max
    z = []
    for ds in datasets:
        sub = df[df["dataset"] == ds].sort_values("alpha")
        vals = sub[metric].values
        z.append((vals / vals.max()) * 100)

    z = np.array(z)

    # Find column-wise minimum across all datasets (worst-case at each alpha)
    worst_case = z.min(axis=0)

    fig = go.Figure()

    fig.add_trace(go.Heatmap(
        z=z,
        x=[f"{a:.2f}" for a in alphas],
        y=list(datasets),
        colorscale=[
            [0.0, "#d73027"],   # red  – worst
            [0.3, "#fc8d59"],   # orange
            [0.5, "#fee08b"],   # yellow
            [0.7, "#d9ef8b"],   # lime
            [0.85, "#66bd63"],  # green
            [1.0, "#1a9850"],   # dark green – best
        ],
        zmin=max(np.floor(z.min()) - 1, 0),  # dynamic floor just below worst value
        zmax=100,
        text=np.round(z, 1),
        # texttemplate="%{text:.0f}%", #NO TEXT
        textfont=dict(size=9),
        colorbar=dict(title="% of best", ticksuffix="%"),
        hovertemplate="Dataset: %{y}<br>α = %{x}<br>%{z:.1f}% of best<extra></extra>",
    ))

    fig.update_layout(
        title=dict(text=f"<b>{metric}</b> — Relative Performance per Dataset", font_size=16),
        xaxis=dict(title="α", tickangle=-45),
        yaxis=dict(autorange="reversed"),
        width=900, height=300,
        margin=dict(t=60, b=60),
        font=dict(family="Helvetica Neue, Segoe UI, sans-serif"),
    )
    fig.show()


for m in METRICS:
    make_heatmap(df, m, COLORS)


In [30]:
def make_robustness_band(df, metric, colors, zoom=False, fine_tuned_alpha=None):
    alphas = sorted(df["alpha"].unique())
    
    # Normalise each dataset to its own max
    norm_scores = {}
    for ds in df["dataset"].unique():
        sub = df[df["dataset"] == ds].sort_values("alpha")
        vals = sub[metric].values
        norm_scores[ds] = vals / vals.max()
    
    mat = np.array(list(norm_scores.values()))   # shape: (n_datasets, n_alphas)
    mean = mat.mean(axis=0)
    std  = mat.std(axis=0)
    worst = mat.min(axis=0)
    
    best_robust_idx = np.argmax(worst)
    best_robust_alpha = alphas[best_robust_idx]
    
    fig = go.Figure()
    
    # ±1 std band
    fig.add_trace(go.Scatter(
        x=list(alphas) + list(alphas)[::-1],
        y=list(mean + std) + list(mean - std)[::-1],
        fill="toself",
        fillcolor="rgba(99,110,250,0.15)",
        line=dict(width=0),
        name="±1 σ band",
        hoverinfo="skip",
    ))
    
    # Mean line
    fig.add_trace(go.Scatter(
        x=list(alphas), y=list(mean),
        mode="lines", name="Mean (normalised)",
        line=dict(color="#636EFA", width=3),
    ))
    
    # Worst-case line
    fig.add_trace(go.Scatter(
        x=list(alphas), y=list(worst),
        mode="lines", name="Worst-case dataset",
        line=dict(color="#EF553B", width=2, dash="dot"),
    ))
    
    # Optimal robust alpha marker (from test data)
    fig.add_vline(x=best_robust_alpha, line_dash="dash", line_color="#2D6A4F",
                  annotation_text=f"Most robust α = {best_robust_alpha:.2f}",
                  annotation_position="top right",
                  annotation_font_size=11)
    
    # Fine-tuned alpha from training split
    if fine_tuned_alpha is not None:
        fig.add_vline(x=fine_tuned_alpha, line_dash="dot", line_color="#F4831F",
                      annotation_text=f"Fine-tuned α = {fine_tuned_alpha:.2f}",
                      annotation_position="top left",
                      annotation_font_size=11)
    
    # Y-axis range
    if zoom:
        y_floor = max(0, np.floor((mat.min()) * 100 - 2) / 100)  # 2% padding below lowest value
        y_range = [y_floor, 1.02]
        title_suffix = " (zoomed)"
    else:
        y_range = [0, 1.05]
        title_suffix = ""
    
    fig.update_layout(
        title=dict(text=f"<b>{metric}</b> — Robustness across Datasets{title_suffix}", font_size=16),
        xaxis=dict(title="α", dtick=0.1, range=[0, 1]),
        yaxis=dict(title="Normalised score (fraction of per-dataset max)", range=y_range),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        width=820, height=420,
        margin=dict(t=80),
        font=dict(family="Helvetica Neue, Segoe UI, sans-serif"),
    )
    fig.show()
    
    if not zoom:
        print(f"  → Most robust α for {metric}: {best_robust_alpha:.2f} "
              f"(worst-case normalised = {worst[best_robust_idx]:.3f})")


for m in METRICS:
    make_robustness_band(df, m, COLORS, fine_tuned_alpha=fine_tuned_alpha_value)

  → Most robust α for RR@10: 0.35 (worst-case normalised = 0.990)


  → Most robust α for nDCG@10: 0.35 (worst-case normalised = 0.995)


  → Most robust α for AP@100: 0.32 (worst-case normalised = 0.990)


## 3b · Robustness Band (zoomed)

Same data as above, but the Y-axis is scaled to the actual value range so you can see the differences between α values clearly.

The **orange dotted** vertical line marks `fine_tuned_alpha_value` (tuned on training data); the **green dashed** line marks the most robust α derived from test data. Their proximity indicates how well training-set tuning generalises.

In [31]:
for m in METRICS:
    make_robustness_band(df, m, COLORS, zoom=True, fine_tuned_alpha=fine_tuned_alpha_value)

## 4 · Global-α Regret Analysis

**Question:** *If we commit to a single global α, how much does each dataset lose compared to its own optimum?*

For a chosen global α we compute:

$$\text{regret}_{d} = \frac{\text{score}_{d}(\alpha^*_d) - \text{score}_{d}(\alpha_{\text{global}})}{\text{score}_{d}(\alpha^*_d)}$$

This is shown as a grouped bar chart for each metric.  
The global α is chosen as the one maximising average nDCG@10 (you can change this).


In [32]:
def find_global_alpha(df, opt_metric="nDCG@10"):
    """Pick the alpha that maximises mean `opt_metric` across datasets."""
    mean_by_alpha = df.groupby("alpha")[opt_metric].mean()
    return mean_by_alpha.idxmax()


def make_regret_chart(df, global_alpha, metrics, colors):
    datasets = df["dataset"].unique()
    
    fig = make_subplots(
        rows=1, cols=len(metrics),
        subplot_titles=[f"<b>{m}</b>" for m in metrics],
        shared_yaxes=True,
        horizontal_spacing=0.06,
    )
    
    for col_idx, m in enumerate(metrics, 1):
        for ds in datasets:
            sub = df[df["dataset"] == ds]
            best_score = sub[m].max()
            global_score = sub.loc[sub["alpha"].round(2) == round(global_alpha, 2), m].values[0]
            regret = (best_score - global_score) / best_score * 100
            
            fig.add_trace(go.Bar(
                x=[ds], y=[regret],
                marker_color=colors[ds],
                name=ds,
                showlegend=(col_idx == 1),
                hovertemplate=f"<b>{ds}</b><br>Regret: %{{y:.2f}}%<br>"
                              f"Best {m}: {best_score:.4f}<br>"
                              f"At global α: {global_score:.4f}<extra></extra>",
            ), row=1, col=col_idx)
    
    fig.update_yaxes(title_text="Regret (%)", row=1, col=1)
    fig.update_layout(
        title=dict(
            text=f"<b>Regret when using global α = {global_alpha:.2f}</b>  "
                 f"<span style='font-size:12px;color:#666'>(chosen by max mean nDCG@10)</span>",
            font_size=16,
        ),
        barmode="group",
        width=900, height=400,
        margin=dict(t=80),
        font=dict(family="Helvetica Neue, Segoe UI, sans-serif"),
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    )
    fig.show()
    
    return global_alpha


global_alpha = find_global_alpha(df, "nDCG@10")
print(f"Global α (max mean nDCG@10): {global_alpha:.2f}\n")
make_regret_chart(df, global_alpha, METRICS, COLORS)


Global α (max mean nDCG@10): 0.30



0.3

## 5 · Summary Table — Optimal α per Dataset and Metric


In [33]:
summary_rows = []
for ds in df["dataset"].unique():
    sub = df[df["dataset"] == ds]
    row = {"Dataset": ds}
    for m in METRICS:
        best_idx = sub[m].idxmax()
        row[f"Best α ({m})"] = sub.loc[best_idx, "alpha"]
        row[f"Best {m}"]     = sub.loc[best_idx, m]
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.style.format({
    col: "{:.4f}" for col in summary.columns if col != "Dataset"
}).background_gradient(subset=[c for c in summary.columns if "Best α" in c], cmap="YlGnBu")


,Dataset,Best α (RR@10),Best RR@10,Best α (nDCG@10),Best nDCG@10,Best α (AP@100),Best AP@100
0,FiQA,0.0800,0.3826,0.2400,0.3151,0.2400,0.2615
1,SciFact,0.3000,0.6582,0.4400,0.6917,0.3000,0.6524
2,ArguAna,0.4500,0.2354,0.5500,0.3545,0.4500,0.2471
3,NFCorpus,0.2300,0.5541,0.4200,0.3415,0.3600,0.1528


## 6 · Additional Experiment Idea: Leave-One-Domain-Out Robustness

A stronger way to demonstrate α robustness is **leave-one-domain-out (LODO) tuning**:

1. For each target dataset $d$, tune α on the *remaining three* datasets  
   (use their dev/train splits, averaging nDCG@10 to pick the best α).
2. Evaluate that α on $d$'s test set.
3. Compare the LODO-tuned α performance against the *oracle* per-dataset optimum.

This directly answers: *"If my target domain was unseen during tuning, how much do I lose?"*

The snippet below generates the data for this experiment.  
(It assumes you still have all the indexes and BM25 retrievers in memory from the main experiment notebook.)


In [34]:
# ── LODO experiment snippet (run in the main experiment notebook) ──────
# Paste this after all indexes are loaded.

# NOTE: Uncomment and run in the experiment notebook — won't run here
#       since we don't have PyTerrier / indexes loaded.

'''
import numpy as np
import pandas as pd
from fast_forward.util.pyterrier import FFScore, FFInterpolate

fixed_k = 100
alphas = np.round(np.arange(0.0, 1.05, 0.05), 2)

# ── Datasets config (same as sweep) ──
configs = [
    {"name": "FiQA",     "bm25": bm25_fiqa,     "ff_index": fiqa_index,
     "testset": testset_fiqa,  "tuneset": pt.get_dataset("irds:beir/fiqa/dev")},
    {"name": "SciFact",  "bm25": bm25_scifact,   "ff_index": scifact_index,
     "testset": testset_scifact, "tuneset": pt.get_dataset("irds:beir/scifact/train")},
    {"name": "NFCorpus", "bm25": bm25_nfcorpus,  "ff_index": nfcorpus_index,
     "testset": testset_nfcorpus, "tuneset": pt.get_dataset("irds:beir/nfcorpus/dev")},
    {"name": "ArguAna",  "bm25": bm25_arguana,   "ff_index": arguana_index,
     "testset": testset_arguana, "tuneset": testset_arguana},  # only has one split
]

lodo_rows = []
for hold_out_idx, target in enumerate(configs):
    # Tune on the other three
    tune_configs = [c for i, c in enumerate(configs) if i != hold_out_idx]
    
    best_alpha, best_avg = None, -1
    for alpha in alphas:
        scores = []
        for tc in tune_configs:
            ff_score = FFScore(tc["ff_index"])
            ff_int = FFInterpolate(alpha=alpha)
            pipe = (tc["bm25"] % fixed_k) >> ff_score >> ff_int
            res = pt.Experiment(
                [pipe],
                tc["tuneset"].get_topics("text"),
                tc["tuneset"].get_qrels(),
                eval_metrics=["ndcg_cut_10"],
            )
            scores.append(res.iloc[0]["ndcg_cut_10"])
        avg = np.mean(scores)
        if avg > best_avg:
            best_avg = avg
            best_alpha = alpha
    
    # Evaluate on the held-out target
    ff_score = FFScore(target["ff_index"])
    ff_int = FFInterpolate(alpha=best_alpha)
    pipe = (target["bm25"] % fixed_k) >> ff_score >> ff_int
    res = pt.Experiment(
        [pipe],
        target["testset"].get_topics("text"),
        target["testset"].get_qrels(),
        eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
        names=[f"LODO α={best_alpha}"],
    )
    
    lodo_rows.append({
        "target_dataset": target["name"],
        "lodo_alpha": best_alpha,
        "RR@10":   res.iloc[0].get("RR(rel=2)@10", res.iloc[0].get("RR@10")),
        "nDCG@10": res.iloc[0]["nDCG@10"],
        "AP@100":  res.iloc[0].get("AP(rel=2)@100", res.iloc[0].get("AP@100")),
    })
    print(f"[LODO] Held out {target['name']} → tuned α = {best_alpha:.2f}")

lodo_df = pd.DataFrame(lodo_rows)
lodo_df.to_csv(Path.cwd() / "lodo_alpha_results.csv", index=False)
print("\nSaved to lodo_alpha_results.csv")
lodo_df
'''
print("(Snippet for experiment notebook — uncomment to run)")


(Snippet for experiment notebook — uncomment to run)
